In [1]:
!pip install -q haversine category_encoders optuna mapie optuna-integration xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.1/187.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.6 MB/s eta 0:00:00


In [2]:
# --- Libraries

# Standard library
import os
import random
import joblib

# Core scientific stack
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import seaborn as sns

# Geospatial & math utilities
from haversine import haversine, Unit
from pyproj import Proj, Transformer

# Statistical & explanatory analysis
import shap
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scikit-learn: clustering & metrics
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

# Scikit-learn: preprocessing & modeling
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNetCV, Ridge, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.inspection import permutation_importance

# Encoding & gradient boosting frameworks
import category_encoders as ce
from xgboost import XGBRegressor
import lightgbm as lgb

# Prediction intervals
# from mapie.regression import MapieRegressor, MapieQuantileRegressor
# from mapie.metrics import regression_coverage_score, regression_mean_width_score

# Hyperparameter optimization
from optuna.integration import OptunaSearchCV
import optuna.distributions as od
from optuna.samplers import RandomSampler
from dataclasses import dataclass, Field, field
from typing import List, Tuple, Optional, Dict

# Other imports
import re
import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
# Read in the feature engineered dataset
final_dataset_rent = pd.read_csv('processed_nyc_rent_data.csv')

In [5]:
# Explore the final set of features
final_dataset_rent.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6300 entries, 0 to 6299
Data columns (total 48 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      6300 non-null   object 
 1   zipcode                   6300 non-null   int64  
 2   median_rent               6300 non-null   float64
 3   median_list_price         6300 non-null   float64
 4   rent_diff                 6300 non-null   float64
 5   restaurant                6300 non-null   float64
 6   rent_lag1                 6225 non-null   float64
 7   station                   6300 non-null   float64
 8   Unemployed Population     6300 non-null   float64
 9   price_lag1                6225 non-null   float64
 10  median_rent_roll3         6150 non-null   float64
 11  median_rent_roll12        5475 non-null   float64
 12  Median Age                6300 non-null   float64
 13  real_rent                 6300 non-null   float64
 14  rel_pric

In [6]:
# Look at the dataframe
final_dataset_rent.head()

,date,zipcode,median_rent,median_list_price,rent_diff,restaurant,rent_lag1,station,Unemployed Population,price_lag1,median_rent_roll3,median_rent_roll12,Median Age,real_rent,rel_price,avg_sale_to_list,median_list_ppsf,hospital,price_ma_12,Per Capita Income,bank,mall,Total Population,price_ma_6,rent_ma_6,rent_ma_3,median_rent_roll6,rel_rent,state_rent_avg,rent_lag3,median_sale_price_roll12,Median Home Value,park,median_ppsf,bus,median_sale_price_roll6,price_ma_3,median_sale_price_roll3,rent_ma_12,real_price,school,Median Rent,zipcode.1,Total Housing Units,supermarket,CPI,inventory,rent_pct_diff
0,2017-01-31,10001,3922.292665,4135000.0,0.000000,1450.0,NaN,174.0,1029.0,NaN,NaN,NaN,35.8,3922.292665,1.699804,0.978486,2585.751979,56.0,1750000.0,86347.0,258.0,10.0,23332.0,1750000.0,3922.292665,3922.292665,NaN,1.469488,2669.155801,NaN,NaN,460500.0,515.0,1247.500000,8.0,NaN,1.750000e+06,NaN,3922.292665,1.750000e+06,385.0,2114.0,10001,13520.0,135.0,266.917,252.0,0.000000
1,2017-02-28,10001,3863.891996,4600000.0,10.754607,1468.0,3922.292665,175.0,1029.0,1750000.0,NaN,NaN,35.8,3853.137389,1.977328,0.979087,2748.473071,56.0,1882500.0,86347.0,259.0,10.0,23332.0,1882500.0,3893.092330,3893.092330,NaN,1.445119,2673.754335,NaN,NaN,460500.0,511.0,1444.801541,8.0,NaN,1.882500e+06,NaN,3893.092330,2.009392e+06,387.0,2114.0,10001,13520.0,142.0,267.662,264.0,0.278336
2,2017-03-31,10001,3873.435091,3825000.0,9.626336,1495.0,3863.891996,172.0,1029.0,2015000.0,3886.539917,NaN,35.8,3863.808754,1.255630,0.984734,2517.513135,56.0,1680000.0,86347.0,259.0,10.0,23332.0,1680000.0,3886.539917,3886.539917,NaN,1.442181,2685.817507,NaN,NaN,460500.0,510.0,1448.087432,9.0,NaN,1.680000e+06,1.680000e+06,3886.539917,1.271831e+06,387.0,2114.0,10001,13520.0,146.0,267.582,276.0,0.248522
3,2017-04-30,10001,3891.172482,2675000.0,14.972304,1534.0,3873.435091,172.0,1029.0,1275000.0,3876.166523,NaN,35.8,3876.200178,1.465778,0.996564,2176.148047,54.0,1638125.0,86347.0,260.0,12.0,23332.0,1638125.0,3887.698058,3876.166523,NaN,1.439306,2703.506212,3922.292665,NaN,460500.0,508.0,1372.288623,9.0,NaN,1.600833e+06,1.600833e+06,3887.698058,1.506680e+06,386.0,2114.0,10001,13520.0,151.0,267.948,264.0,0.384776
4,2017-05-31,10001,3957.284734,2600000.0,18.680985,1584.0,3891.172482,172.0,1029.0,1512500.0,3907.297436,NaN,35.8,3938.603750,1.243743,0.984016,1975.308642,54.0,1570500.0,86347.0,265.0,12.0,23332.0,1570500.0,3901.615393,3907.297436,NaN,1.451929,2725.535873,3863.891996,NaN,460500.0,508.0,1451.394082,9.0,NaN,1.362500e+06,1.362500e+06,3901.615393,1.293863e+06,385.0,2114.0,10001,13520.0,152.0,268.183,251.0,0.472066


In [7]:
# Drop the Medain Rent column since we want to use the median_rent column as the target variable.
final_dataset_rent = final_dataset_rent.rename(columns={'Median Rent': 'median_rent_annual'})

In [8]:
# Create a copy of the dataframe
df = final_dataset_rent.copy()

In [9]:
# Drop obvious duplicates if present
for col in ['zipcode.1', 'rent_lag1', 'rent_lag3']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Snake-case all column names
def snakify(s: str) -> str:
    s = re.sub(r'[^0-9a-zA-Z]+', ' ', s).strip()
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s)
    s = re.sub(r'\s+', '_', s)
    return s.lower()

df.columns = [snakify(c) for c in df.columns]

In [10]:
base_cols = ['date', 'zipcode']

rent_cols = ['median_rent', 'median_rent_annual', 'rent_diff', 'real_rent', 'rel_rent', 'rent_pct_diff', 'median_rent_roll3', 'median_rent_roll6',
             'median_rent_roll12', 'rent_ma_3', 'rent_ma_6', 'rent_ma_12', 'state_rent_avg']

socio_cols = ['unemployed_population', 'per_capita_income', 'total_population', 'cpi']

poi_cols = ['restaurant', 'station', 'hospital', 'bank', 'mall', 'park', 'bus', 'school', 'supermarket']

price_cols = ['real_price', 'median_list_price', 'median_list_ppsf', 'rel_price', 'price_lag1', 'price_ma_3',  'price_ma_6', 'price_ma_12', 'median_list_ppsf', 'median_ppsf', 'avg_sale_to_list',
              'median_sale_price_roll3', 'median_sale_price_roll6', 'median_sale_price_roll12']

other_home_cols = ['median_age', 'median_home_value', 'total_housing_units', 'inventory']


In [11]:
all_cols = base_cols + rent_cols + price_cols + other_home_cols + socio_cols + poi_cols

In [12]:
# Reorder the columns
df = df[all_cols]

In [13]:
# Basic checks
required = ['date', 'zipcode', 'median_rent']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after cleaning: {missing}")

In [14]:
# -----------------------
# Cleaning & Feature Engineering
# -----------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Standardize zipcode to string for categorical handling
    if "zipcode" in df.columns:
        df["zipcode"] = df["zipcode"].astype(str)
    # Parse date
    if "date" in df.columns:
        # Handle both 'YYYY-MM-DD' and 'YYYY-MM' / 'YYYYM' styles
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        if df["date"].isna().any():
            raise ValueError("Some dates could not be parsed. Please normalize the 'date' column.")
    else:
        raise ValueError("Expected a 'date' column.")
    # Sort for time-based ops
    df = df.sort_values(["zipcode", "date"]).reset_index(drop=True)
    return df

# Write a function to build additional time series features
def build_time_features(df: pd.DataFrame) -> pd.DataFrame:
    dt = df["date"]
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["quarter"] = dt.dt.quarter
    # Seasonality (cyclical encoding)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)
    return df

In [15]:
# Build time features first (safe)
df = clean_columns(df)
df = build_time_features(df)

In [16]:
# Target (t+1)
df['y_next'] = df.groupby('zipcode', group_keys=False)['median_rent'].shift(-1)

In [17]:
# 4) Rent history (lags & rollings)
for k in [1, 3, 12]:
    df[f'median_rent_lag{k}'] = df.groupby('zipcode', group_keys=False)['median_rent'].shift(k)

In [18]:
# Lag key exogenous variables (safe to use at prediction time)
exog_candidates = [
    'median_list_price', 'median_ppsf', 'inventory', 'avg_sale_to_list',
    'cpi', 'unemployed_population', 'per_capita_income', 'state_rent_avg'
]
for col in exog_candidates:
    if col in df.columns:
        df[f'{col}_lag1'] = df.groupby('zipcode', group_keys=False)[col].shift(1)

In [19]:
df.head(2)

,date,zipcode,median_rent,median_rent_annual,rent_diff,real_rent,rel_rent,rent_pct_diff,median_rent_roll3,median_rent_roll6,median_rent_roll12,rent_ma_3,rent_ma_6,rent_ma_12,state_rent_avg,real_price,median_list_price,median_list_ppsf,rel_price,price_lag1,price_ma_3,price_ma_6,price_ma_12,median_list_ppsf,median_ppsf,avg_sale_to_list,median_sale_price_roll3,median_sale_price_roll6,median_sale_price_roll12,median_age,median_home_value,total_housing_units,inventory,unemployed_population,per_capita_income,total_population,cpi,restaurant,station,hospital,bank,mall,park,bus,school,supermarket,year,month,quarter,month_sin,month_cos,y_next,median_rent_lag1,median_rent_lag3,median_rent_lag12,median_list_price_lag1,median_ppsf_lag1,inventory_lag1,avg_sale_to_list_lag1,cpi_lag1,unemployed_population_lag1,per_capita_income_lag1,state_rent_avg_lag1
0,2017-01-31,10001,3922.292665,2114.0,0.000000,3922.292665,1.469488,0.000000,NaN,NaN,NaN,3922.292665,3922.292665,3922.292665,2669.155801,1.750000e+06,4135000.0,2585.751979,1.699804,NaN,1750000.0,1750000.0,1750000.0,2585.751979,1247.500000,0.978486,NaN,NaN,NaN,35.8,460500.0,13520.0,252.0,1029.0,86347.0,23332.0,266.917,1450.0,174.0,56.0,258.0,10.0,515.0,8.0,385.0,135.0,2017,1,1,0.500000,0.866025,3863.891996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-02-28,10001,3863.891996,2114.0,10.754607,3853.137389,1.445119,0.278336,NaN,NaN,NaN,3893.092330,3893.092330,3893.092330,2673.754335,2.009392e+06,4600000.0,2748.473071,1.977328,1750000.0,1882500.0,1882500.0,1882500.0,2748.473071,1444.801541,0.979087,NaN,NaN,NaN,35.8,460500.0,13520.0,264.0,1029.0,86347.0,23332.0,267.662,1468.0,175.0,56.0,259.0,10.0,511.0,8.0,387.0,142.0,2017,2,1,0.866025,0.500000,3873.435091,3922.292665,NaN,NaN,4135000.0,1247.5,252.0,0.978486,266.917,1029.0,86347.0,2669.155801


In [20]:
# ---------- 6) Ratios, densities, YoY, interaction ----------
def build_ratio_and_density_features_zipaware(df_in: pd.DataFrame, target: str = 'median_rent') -> pd.DataFrame:
    """
    Zip-aware feature builder.
    Assumes df has columns: 'zipcode', 'date' (monthly), target (e.g., 'median_rent'), and candidate columns
    already in snake_case. Computes:
      - price_to_rent
      - POI per-capita densities
      - inventory_per_100_units
      - YoY changes (within zip)
      - Within-zip anomalies vs PAST 12-month rolling baselines (time-safe)
      - A simple interaction (income × price_to_rent)
    """
    import numpy as np
    import pandas as pd

    out = df_in.copy()
    out = out.sort_values(['zipcode', 'date'])

    # ---------- helpers ----------
    def roll_mean_past(s: pd.Series, w: int) -> pd.Series:
        # mean over the PAST w months up to t-1 (time-safe)
        return s.shift(1).rolling(window=w, min_periods=w).mean()

    # ---------- A) Affordability: price-to-rent (row-level at zip-month) ----------
    if {'median_list_price', 'median_rent_annual'}.issubset(out.columns):
        denom = out['median_rent_annual'].replace({0: np.nan})
        out['price_to_rent'] = out['median_list_price'] / denom

    # ---------- B) Amenities per 1k people (row-level at zip-month) ----------
    pop_col = next((c for c in ['total_population', 'population'] if c in out.columns), None)
    if pop_col:
        poi_cols = [c for c in ['restaurant','hospital','bank','mall','park','bus','school','supermarket','station'] if c in out.columns]
        for c in poi_cols:
            out[f'{c}_per_1k'] = out[c] / (out[pop_col] / 1000.0).replace({0: np.nan})

    # ---------- C) Inventory per 100 housing units ----------
    if 'inventory' in out.columns:
        if 'total_housing_units' in out.columns:
            out['inventory_per_100_units'] = out['inventory'] / (out['total_housing_units'] / 100.0).replace({0: np.nan})
        else:
            out['inventory_per_100_units'] = out['inventory']

    # ---------- D) YoY changes within each zip (time-safe) ----------
    for base_col in [target, 'median_list_price', 'cpi']:
        if base_col in out.columns:
            out[f'{base_col}_yoy'] = out.groupby('zipcode', group_keys=False)[base_col].pct_change(12)

    # ---------- E) Within-zip anomalies vs past rolling baselines (time-safe) ----------
    # Rent anomaly vs past 12M mean (if enough history)
    #   rent_rel_ma12 > 0  => rent is above its own past-year average for this zip
    out['rent_rel_ma12'] = np.where(out['median_rent_roll12'].notna(),
                                    out[target] / out['median_rent_roll12'] - 1.0,
                                    np.nan)

    # Price-to-rent anomaly vs past 12M mean (if price_to_rent exists)
    if 'price_to_rent' in out.columns:
        out['ptr_ma12_past'] = out.groupby('zipcode', group_keys=False)['price_to_rent'].apply(lambda s: roll_mean_past(s, 12))
        out['ptr_anom12'] = np.where(out['ptr_ma12_past'].notna(),
                                     out['price_to_rent'] / out['ptr_ma12_past'] - 1.0,
                                     np.nan)

    # Inventory per 100 units anomaly vs past 12M mean
    if 'inventory_per_100_units' in out.columns:
        out['inv100_ma12_past'] = out.groupby('zipcode', group_keys=False)['inventory_per_100_units'].apply(lambda s: roll_mean_past(s, 12))
        out['inv100_anom12'] = np.where(out['inv100_ma12_past'].notna(),
                                        out['inventory_per_100_units'] / out['inv100_ma12_past'] - 1.0,
                                        np.nan)

    # ---------- F) Simple interaction ----------
    if 'per_capita_income' in out.columns and 'price_to_rent' in out.columns:
        out['income_x_ptr'] = out['per_capita_income'] * out['price_to_rent']

    return out

df = build_ratio_and_density_features_zipaware(df, target='median_rent')

In [21]:
new_feats = [
    'price_to_rent', 'inventory_per_100_units',
    'median_rent_yoy', 'median_list_price_yoy', 'cpi_yoy',
    'rent_rel_ma12', 'ptr_anom12', 'inv100_anom12',
    'income_x_ptr',
    'restaurant_per_1k','hospital_per_1k','bank_per_1k','mall_per_1k',
    'park_per_1k','bus_per_1k','school_per_1k','supermarket_per_1k','station_per_1k'
]
for col in [c for c in new_feats if c in df.columns]:
    df[f'{col}_lag1'] = df.groupby('zipcode', group_keys=False)[col].shift(1)

In [22]:
# Minimal requirement: we need a known target (y_next) and at least rent_lag1 present
df_model = df.dropna(subset=['y_next', 'median_rent_lag1']).copy()

In [23]:
# Prepare a handy feature list for later modeling ----------

# Start with robust, always-safe features:
base_features = ['median_rent_lag1','median_rent_lag3','median_rent_lag12','median_rent_roll3','median_rent_roll6', 'median_rent_roll12', 'month_sin','month_cos']
# Add ALL *_lag1 features we just created (exogenous, ratios, densities, yoy, interactions)
lag1_features = [c for c in df_model.columns if c.endswith('_lag1')]
train_feature_cols = sorted(set(base_features + lag1_features))

In [25]:
import numpy as np
import pandas as pd
from math import sqrt

# Safety: ensure monthly date and sorting (should already be true from Step 1)
df2 = df_model.copy()

In [26]:
df2 = df2.sort_values(['zipcode','date'])
# Add rent_lag11 if missing (for seasonal naive)
if 'median_rent_lag11' not in df2.columns:
    df2['median_rent_lag11'] = df2.groupby('zipcode', group_keys=False)['median_rent'].shift(11)

# ---------- Helpers: metrics ----------
def _pair_dropna(y_true: pd.Series, y_pred: pd.Series):
    m = y_true.notna() & y_pred.notna()
    return y_true[m].values.astype(float), y_pred[m].values.astype(float)

def mae(y_true, y_pred):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    return float(np.mean(np.abs(yt - yp))) if len(yt) else np.nan

def rmse(y_true, y_pred):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    return float(sqrt(np.mean((yt - yp) ** 2))) if len(yt) else np.nan

def smape(y_true, y_pred, eps=1e-8):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    denom = np.abs(yt) + np.abs(yp)
    sm = np.where(denom > eps, 2.0 * np.abs(yt - yp) / denom, 0.0)
    return float(np.mean(sm)) if len(yt) else np.nan  # in [0, 2]; multiply by 100 later if you want %

def wape(y_true, y_pred, eps=1e-8):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    num = np.sum(np.abs(yt - yp))
    den = np.sum(np.abs(yt))
    return float(num / (den + eps)) if len(yt) else np.nan

# ---------- Helper: make walk-forward folds ----------
def make_time_folds(
    df: pd.DataFrame,
    date_col: str = 'date',
    min_train_months: int = 24,   # ensure enough past for lags/YoY
    val_months: int = 6,          # length of each validation window (months)
    max_folds: int | None = None  # limit number of folds if you want
) -> pd.DataFrame:
    """Return a DataFrame with (fold, train_start, train_end, val_start, val_end) monthly boundaries."""
    months = np.array(sorted(pd.to_datetime(df['date'].dt.to_period('M').unique().to_timestamp()).astype('datetime64[ns]')))
    n = len(months)
    if n < (min_train_months + 1):
        raise ValueError(f"Not enough months ({n}) for min_train_months={min_train_months}.")

    folds = []
    start_train_idx = 0
    train_end_idx = min_train_months - 1

    fold_idx = 1
    while True:
        val_start_idx = train_end_idx + 1
        val_end_idx = val_start_idx + val_months - 1
        if val_end_idx >= n:
            break

        folds.append({
            'fold': fold_idx,
            'train_start': months[start_train_idx],
            'train_end':   months[train_end_idx],
            'val_start':   months[val_start_idx],
            'val_end':     months[val_end_idx],
        })

        # advance: expanding window (grow train; slide val)
        train_end_idx = val_end_idx
        fold_idx += 1
        if max_folds is not None and fold_idx > max_folds:
            break

    return pd.DataFrame(folds)

# Choose fold settings (feel free to adjust)
folds_df = make_time_folds(df2, min_train_months=24, val_months=6, max_folds=None)
print("Folds:")
display(folds_df)

# ---------- Evaluate baselines on each fold ----------
records = []
pred_rows = []

for _, f in folds_df.iterrows():
    tr_end = pd.Timestamp(f['train_end'])
    v_start = pd.Timestamp(f['val_start'])
    v_end   = pd.Timestamp(f['val_end'])

    df_val = df2[(df2['date'] >= v_start) & (df2['date'] <= v_end)].copy()

    # Baseline predictions (computed at time t to forecast t+1 = y_next)
    df_val['y_true'] = df_val['y_next']
    df_val['yhat_naive_last'] = df_val['median_rent']           # y_{t}
    df_val['yhat_seasonal']   = df_val['median_rent_lag11']            # y_{t-11}  (for y_{t+1} seasonal naive)
    # 3-month rolling mean at time t (already known at t)
    if 'median_rent_roll3' not in df_val.columns:
        df_val['median_rent_roll3'] = df2.groupby('zipcode')['median_rent'].transform(lambda s: s.rolling(3, min_periods=3).mean())
        df_val = df_val.copy()
    df_val['yhat_rollmean3'] = df_val['median_rent_roll3']

    # Compute metrics
    metrics = {
        'fold': int(f['fold']),
        'train_end': tr_end.date(),
        'val_start': v_start.date(),
        'val_end':   v_end.date(),
    }
    for name, col in [('naive_last','yhat_naive_last'),
                      ('seasonal','yhat_seasonal'),
                      ('rollmean3','yhat_rollmean3')]:
        yt = df_val['y_true']
        yp = df_val[col]
        metrics[f'{name}_MAE']  = mae(yt, yp)
        metrics[f'{name}_RMSE'] = rmse(yt, yp)
        metrics[f'{name}_sMAPE'] = smape(yt, yp)   # 0..2; multiply by 100 for %
        metrics[f'{name}_WAPE']  = wape(yt, yp)    # 0..1; multiply by 100 for %
    records.append(metrics)

    # Store row-level predictions (optional but handy for plots later)
    keep_cols = ['zipcode','date','y_true','yhat_naive_last','yhat_seasonal','yhat_rollmean3']
    pred_rows.append(df_val[keep_cols].assign(fold=int(f['fold'])))

baseline_preds = pd.concat(pred_rows, ignore_index=True)
baseline_summary = pd.DataFrame.from_records(records)

# ---------- Add an "Overall" row (across all folds) ----------
overall = {'fold': 'overall', 'train_end': None, 'val_start': None, 'val_end': None}
for name in ['naive_last','seasonal','rollmean3']:
    yt = baseline_preds['y_true']
    yp = baseline_preds[f'yhat_{name}']
    overall[f'{name}_MAE']  = mae(yt, yp)
    overall[f'{name}_RMSE'] = rmse(yt, yp)
    overall[f'{name}_sMAPE'] = smape(yt, yp)
    overall[f'{name}_WAPE']  = wape(yt, yp)
baseline_summary = pd.concat([baseline_summary, pd.DataFrame([overall])], ignore_index=True)

# Nicely formatted view (percent columns)
def _fmt_percent(x):
    return None if pd.isna(x) else f"{100*x:,.2f}%"
view = baseline_summary.copy()
for c in [col for col in view.columns if col.endswith('_sMAPE') or col.endswith('_WAPE')]:
    view[c] = view[c].apply(_fmt_percent)

print("\n=== Baseline metrics by fold (MAE/RMSE in currency units; sMAPE/WAPE as %) ===")
print(view.to_string(index=False))

# Peek at predictions (optional)
print("\nPredictions sample:")
display(baseline_preds.head(5))

Folds:


,fold,train_start,train_end,val_start,val_end
0,1,2017-02-01,2019-01-01,2019-02-01,2019-07-01
1,2,2017-02-01,2019-07-01,2019-08-01,2020-01-01
2,3,2017-02-01,2020-01-01,2020-02-01,2020-07-01
3,4,2017-02-01,2020-07-01,2020-08-01,2021-01-01
4,5,2017-02-01,2021-01-01,2021-02-01,2021-07-01
5,6,2017-02-01,2021-07-01,2021-08-01,2022-01-01
6,7,2017-02-01,2022-01-01,2022-02-01,2022-07-01
7,8,2017-02-01,2022-07-01,2022-08-01,2023-01-01
8,9,2017-02-01,2023-01-01,2023-02-01,2023-07-01



=== Baseline metrics by fold (MAE/RMSE in currency units; sMAPE/WAPE as %) ===
   fold  train_end  val_start    val_end  naive_last_MAE  naive_last_RMSE naive_last_sMAPE naive_last_WAPE  seasonal_MAE  seasonal_RMSE seasonal_sMAPE seasonal_WAPE  rollmean3_MAE  rollmean3_RMSE rollmean3_sMAPE rollmean3_WAPE
      1 2019-01-01 2019-02-01 2019-07-01       33.368194        94.316005            1.25%           1.17%     90.807384     123.068800          3.21%         3.18%      52.737440       91.929898           1.93%          1.85%
      2 2019-07-01 2019-08-01 2020-01-01       19.410331        25.319574            0.70%           0.67%    106.493786     123.218555          3.79%         3.68%      29.205261       38.174553           1.02%          1.01%
      3 2020-01-01 2020-02-01 2020-07-01       36.243962        91.951957            1.28%           1.26%     86.456913     134.811179          3.05%         3.00%      50.543594       90.222662           1.77%          1.76%
      4 2020

,zipcode,date,y_true,yhat_naive_last,yhat_seasonal,yhat_rollmean3,fold
0,10001,2019-02-28,4005.202868,3975.544046,3852.908062,3994.812936,1
1,10001,2019-03-31,4060.624706,4005.202868,3924.194804,3989.970125,1
2,10001,2019-04-30,4135.450043,4060.624706,4015.853670,4013.790540,1
3,10001,2019-05-31,4190.172163,4135.450043,4065.071516,4067.092539,1
4,10001,2019-06-30,4207.135516,4190.172163,4079.525081,4128.748970,1


In [29]:
# ---- Prep data ----
dfg = df_model.copy().sort_values(['zipcode','date'])
# ensure positive for logs
dfg = dfg[(dfg['y_next'] > 0) & (dfg['median_rent'] > 0)].copy()
dfg['log_y_t'] = np.log(dfg['median_rent'])
dfg['y_growth'] = np.log(dfg['y_next']) - dfg['log_y_t']   # target

# Features: reuse the safe set from Step 1
numeric_features = list(train_feature_cols)    # rent_lag*, rent_ma_*, month_sin/cos, *_lag1, etc.
categorical_features = ['zipcode']             # global model with zip fixed effects

# OneHot compatibility across sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore')

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', ohe, categorical_features),
    ],
    remainder='drop'
)

def make_pipeline(alpha: float) -> Pipeline:
    return Pipeline(steps=[
        ('prep', preprocess),
        ('model', Ridge(alpha=alpha))
    ])

# ---- Metrics (same as Step 2) ----
def _pair_dropna(y_true: pd.Series, y_pred: pd.Series):
    m = y_true.notna() & y_pred.notna()
    return y_true[m].values.astype(float), y_pred[m].values.astype(float)

def mae(y_true, y_pred):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    return float(np.mean(np.abs(yt - yp))) if len(yt) else np.nan

def rmse(y_true, y_pred):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    return float(np.sqrt(np.mean((yt - yp) ** 2))) if len(yt) else np.nan

def smape(y_true, y_pred, eps=1e-8):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    denom = np.abs(yt) + np.abs(yp)
    sm = np.where(denom > eps, 2.0 * np.abs(yt - yp) / denom, 0.0)
    return float(np.mean(sm)) if len(yt) else np.nan

def wape(y_true, y_pred, eps=1e-8):
    yt, yp = _pair_dropna(pd.Series(y_true), pd.Series(y_pred))
    return float(np.sum(np.abs(yt - yp)) / (np.sum(np.abs(yt)) + eps)) if len(yt) else np.nan

# ---- Train/eval across folds ----
alphas = [0.1, 1.0, 10.0, 100.0]

growth_records = []
growth_pred_rows = []
growth_models = {}

for _, fold in folds_df.iterrows():
    tr_start = pd.Timestamp(fold['train_start'])
    tr_end   = pd.Timestamp(fold['train_end'])
    v_start  = pd.Timestamp(fold['val_start'])
    v_end    = pd.Timestamp(fold['val_end'])
    fid      = int(fold['fold'])

    train_mask = (dfg['date'] >= tr_start) & (dfg['date'] <= tr_end)
    val_mask   = (dfg['date'] >= v_start) & (dfg['date'] <= v_end)

    X_train = dfg.loc[train_mask, numeric_features + categorical_features]
    y_train = dfg.loc[train_mask, 'y_growth']   # NOTE: growth target
    X_val   = dfg.loc[val_mask, numeric_features + categorical_features]
    y_val   = dfg.loc[val_mask, 'y_next']
    log_y_t_val = dfg.loc[val_mask, 'log_y_t']  # offset term

    best = {'alpha': None, 'rmse': np.inf, 'pipe': None, 'yhat': None}

    for a in alphas:
        pipe = make_pipeline(alpha=a)
        pipe.fit(X_train, y_train)                  # learn growth
        delta_hat = pipe.predict(X_val)             # predicted growth
        yhat = np.exp(log_y_t_val.values + delta_hat)  # add offset, back to dollars

        cur_rmse = rmse(y_val, yhat)
        if cur_rmse < best['rmse']:
            best.update({'alpha': a, 'rmse': cur_rmse, 'pipe': pipe, 'yhat': yhat})

    # Record metrics for best alpha
    yhat = best['yhat']
    rec = {
        'fold': fid,
        'alpha': best['alpha'],
        'train_end': tr_end.date(),
        'val_start': v_start.date(),
        'val_end': v_end.date(),
        'MAE': mae(y_val, yhat),
        'RMSE': rmse(y_val, yhat),
        'sMAPE': smape(y_val, yhat),
        'WAPE': wape(y_val, yhat),
    }
    growth_records.append(rec)

    # Row-level preds
    growth_pred_rows.append(pd.DataFrame({
        'fold': fid,
        'zipcode': dfg.loc[val_mask, 'zipcode'].values,
        'date': dfg.loc[val_mask, 'date'].values,
        'y_true': y_val.values,
        'yhat_ridge_growth': yhat
    }))

    growth_models[f'fold_{fid}'] = best['pipe']

ridge_growth_preds = pd.concat(growth_pred_rows, ignore_index=True)
ridge_growth_summary = pd.DataFrame(growth_records)

# ---- Overall row ----
overall = {
    'fold': 'overall',
    'alpha': None,
    'train_end': None,
    'val_start': None,
    'val_end': None,
    'MAE': mae(ridge_growth_preds['y_true'], ridge_growth_preds['yhat_ridge_growth']),
    'RMSE': rmse(ridge_growth_preds['y_true'], ridge_growth_preds['yhat_ridge_growth']),
    'sMAPE': smape(ridge_growth_preds['y_true'], ridge_growth_preds['yhat_ridge_growth']),
    'WAPE': wape(ridge_growth_preds['y_true'], ridge_growth_preds['yhat_ridge_growth']),
}
ridge_growth_summary = pd.concat([ridge_growth_summary, pd.DataFrame([overall])], ignore_index=True)

# ---- Compare to naïve baseline if available ----
if 'baseline_preds' in globals():
    cmp = pd.merge(
        ridge_growth_preds.rename(columns={'yhat_ridge_growth':'yhat_model'}),
        baseline_preds[['zipcode','date','y_true','yhat_naive_last']].rename(columns={'yhat_naive_last':'yhat_naive'}),
        on=['zipcode','date','y_true'], how='inner'
    )
    wape_model = wape(cmp['y_true'], cmp['yhat_model'])
    wape_naive = wape(cmp['y_true'], cmp['yhat_naive'])
    impr = (wape_naive - wape_model) / wape_naive if wape_naive > 0 else np.nan
    print(f"\nOverall WAPE — Naïve: {wape_naive:.4f} | Growth-Ridge: {wape_model:.4f} | Improvement: {100*impr:.2f}%")
else:
    print("\n(baseline_preds not found — skipping baseline comparison)")

# ---- Pretty print ----
def _fmt_pct(x):
    return None if pd.isna(x) else f"{100*x:,.2f}%"
view = ridge_growth_summary.copy()
for c in ['sMAPE','WAPE']:
    view[c] = view[c].apply(_fmt_pct)
print("\n=== Growth-Ridge metrics by fold (MAE/RMSE in currency; sMAPE/WAPE in %) ===")
print(view.to_string(index=False))


Overall WAPE — Naïve: 0.0133 | Growth-Ridge: 0.0138 | Improvement: -3.92%

=== Growth-Ridge metrics by fold (MAE/RMSE in currency; sMAPE/WAPE in %) ===
   fold  alpha  train_end  val_start    val_end         MAE        RMSE  sMAPE   WAPE
      1    0.1 2019-01-01 2019-02-01 2019-07-01 1352.780570 1461.687948 40.97% 36.83%
      2    1.0 2019-07-01 2019-08-01 2020-01-01 1107.033774 1236.862611 37.73% 29.82%
      3   10.0 2020-01-01 2020-02-01 2020-07-01  613.722370  722.318927 19.67% 16.82%
      4  100.0 2020-07-01 2020-08-01 2021-01-01  676.590658  761.514039 26.54% 21.99%
      5  100.0 2021-01-01 2021-02-01 2021-07-01  883.487196  950.137197 31.90% 26.20%
      6  100.0 2021-07-01 2021-08-01 2022-01-01  972.790250 1147.836802 30.94% 24.22%
      7    0.1 2022-01-01 2022-02-01 2022-07-01 1093.633756 1219.399168 30.61% 25.28%
      8  100.0 2022-07-01 2022-08-01 2023-01-01  736.157370  886.838741 19.30% 16.93%
      9  100.0 2023-01-01 2023-02-01 2023-07-01 1066.530679 1165.844191 2

## Lookforward Forecasts (h=3, 6, 12)

In [33]:
# --- Build targets for h in {3,6,12} ---
HORIZONS = [3,6,12]

def add_targets(df, id_col='zipcode', date_col='date', y='median_rent'):
    g = df.sort_values([id_col, date_col]).copy()
    for h in HORIZONS:
        g[f'y_tplus{h}'] = g.groupby(id_col)[y].shift(-h)
    g = g[(g[y] > 0)].copy()
    g['log_y_t'] = np.log(g[y])
    return g

dfh = add_targets(df2, 'zipcode', 'date', 'median_rent')

# --- Horizon-specific feature sets (adjust if you swap inventory/price choices) ---
BASE = {
  3: ['median_rent_lag1','median_rent_lag3','median_rent_roll3','median_rent_roll6',
      'median_rent_yoy_lag1','month_sin','month_cos','cpi_yoy_lag1',
      'inventory_per_100_units_lag1','median_list_price_yoy_lag1'],
  6: ['median_rent_lag1','median_rent_lag3','median_rent_lag12',
      'median_rent_roll3','median_rent_roll6','median_rent_roll12',
      'median_rent_yoy_lag1','month_sin','month_cos','cpi_yoy_lag1',
      'state_rent_avg_lag1','inv100_anom12_lag1','price_to_rent_lag1'],
  12:['median_rent_lag12','median_rent_roll12','median_rent_yoy_lag1',
      'month_sin','month_cos','cpi_yoy_lag1','state_rent_avg_lag1',
      'inv100_anom12_lag1','price_to_rent_lag1']
}

CAT = ['zipcode']

# --- Walk-forward folds (use your corrected version) ---
folds_df = make_time_folds(dfh, min_train_months=24, val_months=6)

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet  # try ElasticNet first
import numpy as np, pandas as pd

def run_direct_growth(dfh, h, num_cols, cat_cols=['zipcode'],
                      alphas=(0.1,1.0,10.0), l1s=(0.0,0.2,0.5)):
    recs, rows = [], []
    ohe = OneHotEncoder(handle_unknown='ignore')
    prep = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc', StandardScaler())]), num_cols),
        ('cat', ohe, cat_cols)
    ], remainder='drop')

    for _, f in folds_df.iterrows():
        tr_start = pd.Timestamp(f['train_start'])
        tr_end   = pd.Timestamp(f['train_end'])
        va_start = pd.Timestamp(f['val_start'])
        va_end   = pd.Timestamp(f['val_end'])

        # -------- TRAIN: cut h months off the end so y_{t+h} exists within the window
        train_cut = tr_end - pd.DateOffset(months=h)
        tr_mask_h = (dfh['date'] >= tr_start) & (dfh['date'] <= train_cut)

        # Keep only rows with valid, positive targets and current rent
        y_tr_future = dfh.loc[tr_mask_h, f'y_tplus{h}']
        y_tr_now    = dfh.loc[tr_mask_h, 'median_rent']
        valid_tr = y_tr_future.notna() & y_tr_now.notna() & (y_tr_future > 0) & (y_tr_now > 0)

        # Build X/y for train after filtering
        X_tr = dfh.loc[tr_mask_h, num_cols + cat_cols].loc[valid_tr]
        y_tr = (np.log(y_tr_future.loc[valid_tr].values) - np.log(y_tr_now.loc[valid_tr].values))

        # -------- VAL: standard fold window; drop rows where target missing/≤0
        va_mask = (dfh['date'] >= va_start) & (dfh['date'] <= va_end)
        y_va_true = dfh.loc[va_mask, f'y_tplus{h}']
        y_va_now  = dfh.loc[va_mask, 'median_rent']
        valid_va = y_va_true.notna() & y_va_now.notna() & (y_va_true > 0) & (y_va_now > 0)

        X_va        = dfh.loc[va_mask, num_cols + cat_cols].loc[valid_va]
        y_va_true_v = y_va_true.loc[valid_va].values
        log_y_t_va  = np.log(y_va_now.loc[valid_va].values)

        # Safety checks (optional, helpful for debugging)
        if len(X_tr) == 0 or len(X_va) == 0:
            # Not enough data in this fold/horizon — skip gracefully
            continue

        best_rmse, best_alpha, best_l1, best_pred = np.inf, None, None, None
        for a in alphas:
            for l1 in l1s:
                model = Pipeline([
                    ('prep', prep),
                    ('enet', ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000))
                ])
                model.fit(X_tr, y_tr)
                d_hat = model.predict(X_va)
                yhat  = np.exp(log_y_t_va + d_hat)
                rmse  = np.sqrt(np.mean((yhat - y_va_true_v)**2))
                if rmse < best_rmse:
                    best_rmse, best_alpha, best_l1, best_pred = rmse, a, l1, yhat

        recs.append({
            'fold': int(f['fold']), 'h': h,
            'alpha': best_alpha, 'l1_ratio': best_l1,
            'MAE': float(np.mean(np.abs(best_pred - y_va_true_v))),
            'RMSE': float(np.sqrt(np.mean((best_pred - y_va_true_v)**2))),
        })
        rows.append(pd.DataFrame({
            'fold': int(f['fold']), 'h': h,
            'zipcode': dfh.loc[va_mask, 'zipcode'].loc[valid_va].values,
            'date':    dfh.loc[va_mask, 'date'].loc[valid_va].values,
            'y_true':  y_va_true_v,
            'yhat':    best_pred
        }))

    # If a horizon had no valid folds, return empty frames
    if len(recs) == 0:
        return pd.DataFrame(columns=['fold','h','alpha','l1_ratio','MAE','RMSE']), \
               pd.DataFrame(columns=['fold','h','zipcode','date','y_true','yhat'])

    return pd.DataFrame(recs), pd.concat(rows, ignore_index=True)

# --- Run all horizons
all_fold_metrics, all_preds = [], []
for h in [3,6,12]:
    m, r = run_direct_growth(dfh, h, num_cols=BASE[h])
    all_fold_metrics.append(m.assign(h=h))
    all_preds.append(r)
metrics_h = pd.concat(all_fold_metrics, ignore_index=True)
preds_h = pd.concat(all_preds, ignore_index=True)

# Overall by horizon (computed ONLY from concatenated per-fold rows)
overall_by_h = preds_h.groupby('h').apply(
    lambda g: pd.Series({
        'MAE': np.mean(np.abs(g['yhat']-g['y_true'])),
        'RMSE': np.sqrt(np.mean((g['yhat']-g['y_true'])**2))
    })
).reset_index()
print(overall_by_h)

    h         MAE        RMSE
0   3   86.696120  134.043771
1   6  152.700574  226.304799
2  12  250.390580  340.184869


In [34]:
def horizon_baselines(df, h):
    g = df.sort_values(['zipcode','date']).copy()
    # targets
    g[f'y_tplus{h}'] = g.groupby('zipcode')['median_rent'].shift(-h)
    # baselines available at time t
    g[f'naive_h{h}'] = g['median_rent']                      # y_t
    g[f'seasonal_h{h}'] = g.groupby('zipcode')['median_rent'].shift(12-h)  # y_{t+h-12}
    g['roll3_t'] = g.groupby('zipcode')['median_rent'].rolling(3, min_periods=3).mean().reset_index(level=0, drop=True)

    m = g[f'y_tplus{h}'].notna() & g['roll3_t'].notna() & g[f'seasonal_h{h}'].notna()
    yt = g.loc[m, f'y_tplus{h}']
    def MAE(y, p): return float((y - p).abs().mean())
    return {
        'naive_MAE': MAE(yt, g.loc[m, f'naive_h{h}']),
        'seasonal_MAE': MAE(yt, g.loc[m, f'seasonal_h{h}']),
        'roll3_MAE': MAE(yt, g.loc[m, 'roll3_t'])
    }

for h in [3,6,12]:
    print(h, horizon_baselines(df2, h))

3 {'naive_MAE': 89.67394300969258, 'seasonal_MAE': 213.3610565937205, 'roll3_MAE': 107.13601434519086}
6 {'naive_MAE': 144.17340489898962, 'seasonal_MAE': 213.3610565937205, 'roll3_MAE': 154.9889626753953}
12 {'naive_MAE': 218.17146097791203, 'seasonal_MAE': 218.17146097791203, 'roll3_MAE': 228.28459079610724}
